# Credit Card Fraud Classfication with Structured API

## Prerequisites

Starting HDFS server with

In [1]:
!$HADOOP_HOME/sbin/start-dfs.sh
!$HADOOP_HOME/sbin/start-yarn.sh

Starting namenodes on [localhost]
Starting datanodes
Starting secondary namenodes [DESKTOP-OECCCK2]
Starting resourcemanager
Starting nodemanagers


In [3]:
import findspark
findspark.init()

In [4]:
from pyspark.sql import SparkSession
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.feature import StandardScaler, VectorAssembler

spark = SparkSession.builder \
          .appName("Structured_API")\
          .getOrCreate()

25/05/10 16:34:44 WARN Utils: Your hostname, DESKTOP-OECCCK2 resolves to a loopback address: 127.0.1.1; using 172.28.205.125 instead (on interface eth0)
25/05/10 16:34:44 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/05/10 16:34:44 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [5]:
import os

data_path = f"file:///{os.getcwd()}/creditcard.csv"
df = spark.read.csv(data_path, header=True, inferSchema=True)

## Data Preprocessing

### Loại bỏ những dữ liệu cột không phù hợp

In [6]:
from pyspark.sql.functions import count, col, isnan, when, asc, desc, round

In [7]:
# Lọc ra các principle components trong bộ dữ liệu
df = df.drop('Time')
df = df.drop('Amount')


### Kiểm tra dữ liệu rỗng, cho biết số lượng dữ liệu rỗng trong bộ dữ liệu

In [8]:
# Kiểm tra dữ liệu có tồn tại giá trị rỗng
df.select([
    count(when(isnan(col(c)) | col(c).isNull(), c)).alias(c) for c in df.columns
]).show()

25/05/10 16:35:08 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
[Stage 2:====================================>                      (5 + 3) / 8]

+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+-----+
| V1| V2| V3| V4| V5| V6| V7| V8| V9|V10|V11|V12|V13|V14|V15|V16|V17|V18|V19|V20|V21|V22|V23|V24|V25|V26|V27|V28|Class|
+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+-----+
|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|    0|
+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+-----+



### Kiểm tra dữ liệu trùng

In [9]:
# Cho biết các bản ghi có dữ liệu trùng
num_duplicate_data = df.count()-df.distinct().count()
duplicate_ratio = (num_duplicate_data)/df.count() * 100
print(f"Dữ liệu trùng lặp: {num_duplicate_data}")
print(f"Tỉ lệ dữ liệu trùng lặp: {duplicate_ratio:.4f} %")
df = df.distinct()

Dữ liệu trùng lặp: 9144
Tỉ lệ dữ liệu trùng lặp: 3.2106 %


In [10]:
label_column = df.columns[-1]
feature_columns = df.columns[:-1]

assembler = VectorAssembler(inputCols=feature_columns, outputCol='features')
df = assembler.transform(df)

In [11]:
train, test = df.randomSplit([0.8, 0.2], seed=42)

In [12]:
scaler = StandardScaler(inputCol="features",
                        outputCol="scaled_features",
                        withStd=True,
                        withMean=True)
scaler_model = scaler.fit(train)

In [13]:
train = scaler_model.transform(train)
test = scaler_model.transform(test)

## Huấn luyện mô hình

Ta sẽ sử dụng nhiều bộ tham số lr với maxiters để tìm bộ tham số tối ưu hóa hiệu suất mô hình

In [14]:
learning_rates = [1, 0.01, 0.001]
listIters = [20,50,100]

In [15]:
results = []
for lr, maxIter in zip(learning_rates,listIters):
    # Khởi tạo LogisticRegression với learning rate và maxIter khác nhau
    log_reg = LogisticRegression(
        labelCol=label_column,
        featuresCol="scaled_features",
        maxIter=maxIter,
        threshold=0.5,
    )

    # Huấn luyện mô hình
    model = log_reg.fit(train)

    predictions = model.transform(test)

    evaluator_binary = BinaryClassificationEvaluator(labelCol=label_column, rawPredictionCol="rawPrediction")
    auc = evaluator_binary.evaluate(predictions, {evaluator_binary.metricName: "areaUnderROC"})


    evaluator_multi = MulticlassClassificationEvaluator(labelCol=label_column, predictionCol="prediction")

    accuracy = evaluator_multi.evaluate(predictions, {evaluator_multi.metricName: "accuracy"})
    f1_score = evaluator_multi.evaluate(predictions, {evaluator_multi.metricName: "f1"})
    precision = evaluator_multi.evaluate(predictions, {evaluator_multi.metricName: "weightedPrecision"})
    recall = evaluator_multi.evaluate(predictions, {evaluator_multi.metricName: "weightedRecall"})

    TP = predictions.filter((col(label_column) == 1.0) & (col("prediction") == 1.0)).count()
    FP = predictions.filter((col(label_column) == 0.0) & (col("prediction") == 1.0)).count()
    TN = predictions.filter((col(label_column) == 0.0) & (col("prediction") == 0.0)).count()
    FN = predictions.filter((col(label_column) == 1.0) & (col("prediction") == 0.0)).count()

    results.append({
        "lr": lr,
        "maxIter": maxIter,
        "auc": auc,
        "accuracy": accuracy,
        "recall": recall,
        "prediction_result": {
            "TP": TP,
            "TN": TN,
            "FP": FP,
            "FN": FN
        }
    })

25/05/10 16:35:44 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
                                                                                

In [16]:
def check_label_ratio(df):
    """
    Hàm kiểm tra tỉ lệ các nhãn trong PySpark DataFrame.
    Args:
        df: PySpark DataFrame.
    Returns:
        Một danh sách các dictionary, mỗi dictionary chứa:
        - label: Nhãn
        - count: Số lượng mẫu của nhãn
        - ratio: Tỉ lệ của nhãn trong tổng số mẫu
        Hoặc một danh sách rỗng nếu DataFrame đầu vào rỗng.
    """

    label_counts_agg_df = df.groupBy("Class").count()

    collected_counts = label_counts_agg_df.collect()

    # Chuyển đổi thành định dạng dictionary mong muốn cho dễ xử lý
    result = []
    for row in collected_counts:
        label = row["Class"] # Lấy giá trị từ cột nhãn
        count_val = row["count"]        # Lấy giá trị từ cột count (tên mặc định từ .count())
        ratio = (count_val / df.count() * 100)
        result.append({
            "label": label,
            "count": count_val,
            "ratio": f"{ratio:.3f}%"
        })

    result.sort(key=lambda x: x['label'])

    return result

In [17]:
final_result = {
    "train": check_label_ratio(train),
    "test": check_label_ratio(test),
    "model_result": results
}

In [19]:
import json
output_file = "./Structured_API.json"
with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(final_result, f, indent=4, ensure_ascii=False)

print(f"Kết quả đã được lưu vào file: {output_file}")

Kết quả đã được lưu vào file: ./Structured_API.json


In [20]:
spark.stop()
!$HADOOP_HOME/sbin/stop-dfs.sh
!$HADOOP_HOME/sbin/stop-yarn.sh

Stopping namenodes on [localhost]
Stopping datanodes
Stopping secondary namenodes [DESKTOP-OECCCK2]
Stopping nodemanagers
localhost: WARNING: nodemanager did not stop gracefully after 5 seconds: Trying to kill with kill -9
Stopping resourcemanager
